
# 2. Text Preprocessing

**Objective:** Remove duplicate reviews, inspect and clean HTML formatting, and prepare review text for sentiment classification and keyphrase extraction.

We will preserve the original review text and create a separate cleaned-text column for modelling.

In [1]:
import pandas as pd
import re
import html
from pathlib import Path

In [2]:
data_path = Path("../dataset/raw/IMDB Dataset.csv")

df = pd.read_csv(data_path)

print("Original dataset shape:", df.shape)
df.head()

Original dataset shape: (50000, 2)


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
# Removing duplicate reviews
# Our EDA found 418 extra rows containing duplicate reviews, with no conflicting sentiment labels.

print("Rows before deduplication:", len(df))

df = df.drop_duplicates(subset="review").reset_index(drop=True)

print("Rows after deduplication:", len(df))
print("Rows removed:", 50000 - len(df))

Rows before deduplication: 50000
Rows after deduplication: 49582
Rows removed: 418


In [4]:
# Inspecting html and text formatting
for i in range(3):
    print(f"\nReview {i + 1}:")
    print(df.loc[i, "review"][:500])


Review 1:
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ

Review 2:
A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. <br /><br />The actors are extremely well chosen- Michael Sheen not only "has got all the polari" but he has all the voices down pat too! You can truly see the seamless editing guided by the references to Williams' diary entries, not only is it well worth the watching but it

In [5]:
# Text cleaning function to remove line breaks, html tags, and unescape html entities
def clean_review(text):
    # Unescape HTML entities, e.g. &amp; and &lt;
    text = html.unescape(text)

    # Remove HTML tags, e.g. <br /> and <div>
    text = re.sub(r"<[^>]+>", " ", text)

    # Normalise whitespace
    text = re.sub(r"\s+", " ", text).strip()

    # Convert to lowercase
    text = text.lower()

    return text

In [6]:
# Applying the clean review function to the review column
df["cleaned_review"] = df["review"].apply(clean_review)

df[["review", "cleaned_review", "sentiment"]].head(3)

,review,cleaned_review,sentiment
0,One of the other reviewers has mentioned that ...,one of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,a wonderful little production. the filming tec...,positive
2,I thought this was a wonderful way to spend ti...,i thought this was a wonderful way to spend ti...,positive


In [8]:
# Comparing the original and cleaned versions
sample = df.iloc[0]

print("ORIGINAL REVIEW:\n")
print(sample["review"][:700])

print("\nCLEANED REVIEW:\n")
print(sample["cleaned_review"][:700])

ORIGINAL REVIEW:

One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face

CLEANED REVIEW:

one of the other reviewers has mentioned that after watching just 1 oz episode you'll be hooked. they are right, as this is exactly what happened with me. the first thing that struck me about oz was its brutality and unflinching scenes of violence, which set in r

In [12]:
# Checking if any html tags remain
html_tag_count = df["cleaned_review"].str.contains(
    r"<[^>]+>",
    regex=True
).sum()

print("Reviews still containing HTML-like tags:", html_tag_count)

# Check for HTML entities that may still need attention
entity_pattern = r"&(?:amp|lt|gt|quot|apos|nbsp);"

print(
    "Reviews containing common HTML entities:",
    df["cleaned_review"].str.contains(
        entity_pattern, case=False, regex=True
    ).sum()
)

Reviews still containing HTML-like tags: 0
Reviews containing common HTML entities: 0


In [10]:
# Validating the cleaned dataset
print("Dataset shape:", df.shape)

print("\nMissing values:")
print(df[["review", "cleaned_review", "sentiment"]].isnull().sum())

print("\nDuplicate original reviews:", df["review"].duplicated().sum())
print("Empty cleaned reviews:", df["cleaned_review"].str.strip().eq("").sum())

print("\nSentiment distribution:")
print(df["sentiment"].value_counts())

Dataset shape: (49582, 3)

Missing values:
review            0
cleaned_review    0
sentiment         0
dtype: int64

Duplicate original reviews: 0
Empty cleaned reviews: 0

Sentiment distribution:
sentiment
positive    24884
negative    24698
Name: count, dtype: int64


In [11]:
# Saving the processed dataset
output_path = Path("../dataset/processed/imdb_reviews_cleaned.csv")

output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(output_path, index=False)

print(f"Processed dataset saved to: {output_path}")

Processed dataset saved to: ..\dataset\processed\imdb_reviews_cleaned.csv
